# Nodes and Terminals

All current-carrying electrical equipment inheriting from the `ConductingEquipment` class are defined as being connected to a Terminal object associated with a `ConnectivityNode`. Generators, shunt capacitors, shunt reactors, and loads are connected to one Terminal object associated with a single `ConnectivityNode`. Lines, cables, series capacitors, and series reactors have two Terminal objects associated with either end of the branch. Rather than defining from/to buses, CIM identifies the end of a branch by setting the `ACDCTerminal.sequenceNumber` attribute of a terminal to values of 1 or 2 to specify the particular end. Transformers are defined using two or three terminals, with the `ACDCTerminal`. `sequenceNumber` attribute used to designate whether the Terminal is associated with the primary, secondary, or tertiary windings. Split-phase distribution transformers also use three terminals, but the two low-side terminals are associated with a single `ConnectivityNode`, as shown in Figure below. 

![Alt text](images/Fig_8_Chapter_5.png)

The Configuration of `ConnectivityNode` and `Terminal` objects are displayed in this Figure. From left to right, they are `SynchronousMachine`, `PowerElectronicsConnection`, `ShuntCompensator`, `EnergyConsumer`, `ACLineSegment`, `LoadBreakSwitch`, two-winding `PowerTransformer`, three-winding `PowerTransformer`, and split-phase distribution `TransformerTank`. 

Although this approach may seem excessively detailed and complicated compared to bus-branch modeling used by many analysis tools, the set of `ConnectivityNode` and Terminal objects provide a node-edge graph structure built into the power system network model. This graph structure can then be used for highly efficient topology processing and mapping of the electrical network using the `TopologicalNode` and `TopologicalIsland` objects associated with each `ConnectivityNode`. 

Consider a data mapping problem in which DERs need to be associated with substation breakers: The entire analysis could be accomplished with only a topological model specifying the association of `ConnectivityNode` and Terminal objects to their associated DERs, lines, transformers, and switches. The node-edge graph structure built into the CIM model can then be used to determine which DERs are connected to which breakers by building a spanning tree to determine the associated Feeder or `TopologicalIsland` in which the DER is contained. Detailed modeling of other aspects of the power system model (such as would be needed to solve a full power flow solution) is not required, and consequently, a very simple CIM profile could be adopted with far less complexity than would be needed for full model exchange between traditional analysis software packages. The UML class diagram showing the detailed associations between nodes, terminals, power system equipment, and how they can be organized into a particular feeder and substation is shown in in Figure below.

In [1]:
import os
from cimgraph import utils
from mermaid import Mermaid
import cimgraph.data_profile.cimhub_2023 as cim
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'

In [2]:
diagram_text = utils.get_mermaid([cim.Terminal,
                                  cim.ConductingEquipment, 
                                  cim.ConnectivityNode, 
                                  cim.Feeder, 
                                  cim.Substation, 
                                  cim.Equipment])
Mermaid(diagram_text)

The Figure displays a UML class diagram depicting how a piece of ConductingEquipment has a set of associated Terminal objects, which can be associated with a Feeder, which can then be associated with a Substation. Associated with each Terminal object are measurements of power system parameters. A `Measurement` object is used to represent any direct measurement, calculated value, or non-measured non-calculated value. Examples of possible `Measurement` objects include the position of a transformer tap, current measured by a current transformer (CT), calculated MW flow of a line, the oil temperature of a transformer, or whether a substation door is open. The type of a measurement is specified as a text string through the `Measurement.measurementType` attribute. CIM does not directly specify how measurements are named or defined. Rather, it provides high-level classes for the overall type of `Measurement`, such as `Analog` (e.g. voltage), `Discrete` (e.g. breaker status), and `Accumulator` (e.g. metered kWh). `Measurement` objects associated with power flow quantities (e.g. MW flow at the end of a line) are associated with the `ACDCTerminal` at which the measurement is taken. `Measurement` objects relating to properties of the equipment itself (e.g. oil temperature) are generally associated with the `PowerSystemResource` object for that piece of equipment. Figure above illustrates how a Measurement is associated to the `MeasurementValue`, `Terminal`, and `PowerSystemResource`.  Because CIM objects inherit all of associations of their parent classes, the Measurement can be associated with a specific device, such as a `Switch`.

In [3]:
diagram_text = utils.get_mermaid([cim.Terminal, cim.Measurement, cim.PowerSystemResource, cim.ConductingEquipment, cim.ConnectivityNode, cim.ACDCTerminal,  cim.Equipment, cim.Switch, cim.TopologicalNode, cim.TopologicalIsland])
Mermaid(diagram_text)

The Figure displays a UML class diagram showing that every piece of `ConductingEquipment` has a set of `Terminal` objects through which it is connected to the electrical network. Each Terminal has an associated `ConnectivityNode`, which in turn, may have defined a `TopologicalNode` that is used to identify whether the `ConnectivityNode` is contained inside a `TopologicalIsland`. SCADA Measurement objects related to power flow are associated with the equipment terminals, rather than with the equipment object itself.

Figure below highlights the different kinds of measurements available within CIM, including the classes:
* Analog – used for voltage, power, current and other continuous measurements
* Discrete – used for switch open/closed positions, transformer tap positions, etc.
* Accumulator – used for time-integrated measurements (such as meter kWh)


In [4]:
diagram_text = utils.get_mermaid([cim.Measurement,
                                  cim.Analog, cim.Accumulator, cim.StringMeasurement,cim.Discrete])
Mermaid(diagram_text)

The Figure shows UML class diagram depicting the association of a `Measurement` to an `ACDCTerminal` and/or `PowerSystemResource`. A `Measurement` can be `Analog`, `Discrete`, `Accumulator`, or `StringMeasurement`. The numerical value of the measurement is an attribute of `AnalogValue`, `DiscreteValue`, `AccumulatorValue`, or `StringMeasurementValue`, which inherit from the `MeasurementValue` class.

Some examples are explained next.


In [5]:
from cimgraph.databases import XMLFile
from cimgraph.models import FeederModel

In [6]:
xml_file = XMLFile(filename="../sample_models/ieee13.xml")
network = FeederModel(connection=xml_file, container=None)

Example 1: Which topological island is node with mrid 0124E881-B82D-4206-BBDF-37D585159872 part of?

In [7]:
# Retrieve the node with the specified UUID from the graph
node = network.get_object(mRID='0124E881-B82D-4206-BBDF-37D585159872')

# Traverse ConnectivityNode -> TopologicalNode -> TopologicalIsland
topological_island = node.TopologicalNode.TopologicalIsland

print(topological_island.name)

ieee13nodeckt_Island


Example 2: What is the length of the line from node bus 632 to node bus 645?

In [8]:
from_name = '632'
to_name = '645'
results = set()

# find_by_attribute returns all ConnectivityNodes whose name contains the search string
for node in network.find_by_attribute(cim.ConnectivityNode, 'name', from_name):
    for terminal in node.Terminals:
        equipment = terminal.ConductingEquipment
        if isinstance(equipment, cim.ACLineSegment):
            # Check the far end of the line for the target node
            for far_terminal in equipment.Terminals:
                if to_name in far_terminal.ConnectivityNode.name:
                    results.add(equipment.length)

print(results)

{152.4}


Example 3: What types of equipment are connected to node bus 634?

In [9]:
name = '634'
results = []

for node in network.find_by_attribute(cim.ConnectivityNode, 'name', name):
    for terminal in node.Terminals:
        equipment = terminal.ConductingEquipment
        results.append(equipment.__class__.__name__)

print(results)

['PowerElectronicsConnection', 'PowerElectronicsConnection', 'PowerElectronicsConnection', 'PowerTransformer', 'EnergyConsumer', 'EnergyConsumer', 'EnergyConsumer']


Example 4: What is the real power of load connected to bus 634?

In [10]:
name = '634'
results = []

for node in network.find_by_attribute(cim.ConnectivityNode, 'name', name):
    for terminal in node.Terminals:
        equipment = terminal.ConductingEquipment
        if isinstance(equipment, cim.EnergyConsumer):
            results.append(equipment.p)

print(results)

[160000.0, 120000.0, 120000.0]


Example 5: What are the other nodes connected to bus 634?

In [11]:
name = '634'
results = set()

for node in network.find_by_attribute(cim.ConnectivityNode, 'name', name):
    for terminal in node.Terminals:
        equipment = terminal.ConductingEquipment
        # Walk to the far-end node of each connected piece of equipment
        for far_terminal in equipment.Terminals:
            results.add(far_terminal.ConnectivityNode.name)

print(results)

{'634', 'xf1'}


Example 6: What type of equipment connects nodes 670 and house?

In [12]:
from_name = '670'
to_name = 'house'
results = set()

for node in network.find_by_attribute(cim.ConnectivityNode, 'name', from_name):
    for terminal in node.Terminals:
        equipment = terminal.ConductingEquipment
        # Equipment connects both nodes if its far terminal lands on to_name
        for far_terminal in equipment.Terminals:
            if to_name in far_terminal.ConnectivityNode.name:
                results.add(equipment.__class__.__name__)

print(results)

{'PowerTransformer'}


Example 7: What is the uuid of node bus 634?

In [13]:
name = '634'

# find_by_attribute matches on substring; take the first match
node = network.find_by_attribute(cim.ConnectivityNode, 'name', name)[0]

print(node.mRID)

0DCC57AF-F4FA-457D-BB24-2EFDA9865A1A


Example 8: What voltage limits are defined in the model? What is the maximum allowable voltage?

`VoltageLimit` objects are stored in `OperationalLimitSet` collections. In this model the limit sets are not back-referenced from individual nodes, so we list every `VoltageLimit` directly with `list_by_class`.

In [14]:
# list_by_class returns all VoltageLimit objects, sorted and typed
voltage_limits = network.list_by_class(cim.VoltageLimit)

results = [[limit.name, float(limit.value)] for limit in voltage_limits]

print(results)

# Maximum allowable voltage across all limits
print(max(float(limit.value) for limit in voltage_limits))

[['OpLimV_13.2000_RangeAHi', 13860.0], ['OpLimV_0.2080_RangeAHi', 218.4], ['OpLimV_13.2000_RangeBLo', 12100.0], ['OpLimV_0.2080_RangeBHi', 220.13333], ['OpLimV_13.2000_RangeALo', 12540.0], ['OpLimV_0.4800_RangeBLo', 440.00002], ['OpLimV_115.0000_RangeBLo', 105416.67], ['OpLimV_4.1600_RangeALo', 3952.0], ['OpLimV_4.1600_RangeAHi', 4368.0], ['OpLimV_115.0000_RangeBHi', 121708.33], ['OpLimV_4.1600_RangeBLo', 3813.3335], ['OpLimV_0.4800_RangeALo', 456.0], ['OpLimV_0.2080_RangeALo', 197.6], ['OpLimV_0.4800_RangeAHi', 504.0], ['OpLimV_115.0000_RangeAHi', 120750.0], ['OpLimV_13.2000_RangeBHi', 13970.0], ['OpLimV_0.4800_RangeBHi', 507.99998], ['OpLimV_115.0000_RangeALo', 109250.0], ['OpLimV_4.1600_RangeBHi', 4402.6665], ['OpLimV_0.2080_RangeBLo', 190.66667]]
121708.33


Example 9: What phases are associated with bus node 634?

In [17]:
name = 'house'
results = set()

# graph path: ConnectivityNode -> Terminals -> ConductingEquipment -> phases
for node in network.find_by_attribute(cim.ConnectivityNode, 'name', name):
    for terminal in node.Terminals:
        equipment = terminal.ConductingEquipment
        if isinstance(equipment, cim.PowerTransformer):
            for transformer_tank in equipment.TransformerTanks:
                for tank_end in transformer_tank.TransformerTankEnds:
                    # Match the tank end to the terminal on this node
                    if tank_end.Terminal == terminal and tank_end.orderedPhases is not None:
                        results.add(str(tank_end.orderedPhases))

# An empty result typically indicates a three-phase node
if not results:
    results = {'no phases found, it is likely a three-phase node'}

print(results)

{'OrderedPhaseCodeKind.Ns2', 'OrderedPhaseCodeKind.s1N'}


Example 10: What are the xy location coordinates for node bus 634?

In [18]:
name = '634'
results = []

# graph path: ConnectivityNode -> Terminals -> ConductingEquipment -> Location -> PositionPoints
for node in network.find_by_attribute(cim.ConnectivityNode, 'name', name):
    for terminal in node.Terminals:
        equipment = terminal.ConductingEquipment
        if equipment.Location is not None:
            for position_point in equipment.Location.PositionPoints:
                # Match the position point to this terminal's sequenceNumber
                if position_point.sequenceNumber == terminal.sequenceNumber:
                    results.append({'x': position_point.xPosition,
                                    'y': position_point.yPosition})

print(results)

[{'x': '400', 'y': '250'}, {'x': '400', 'y': '250'}, {'x': '400', 'y': '250'}, {'x': '400', 'y': '250'}, {'x': '400', 'y': '250'}, {'x': '400', 'y': '250'}, {'x': '400', 'y': '250'}]


Example 11: What is the name of the inverters connected to node bus 634?

In [19]:
name = '634'
results = []

for node in network.find_by_attribute(cim.ConnectivityNode, 'name', name):
    for terminal in node.Terminals:
        equipment = terminal.ConductingEquipment
        # PowerElectronicsConnection represents inverter-based resources
        if isinstance(equipment, cim.PowerElectronicsConnection):
            results.append(equipment.name)

print(results)

['school', 'school', 'batidle']


Example 12: What is the nominal voltage for node bus 634?


In [20]:
name = '634'
results = set()

# graph path: ConnectivityNode -> Terminals -> ConductingEquipment -> BaseVoltage
for node in network.find_by_attribute(cim.ConnectivityNode, 'name', name):
    for terminal in node.Terminals:
        equipment = terminal.ConductingEquipment
        if equipment.BaseVoltage is not None:
            results.add(equipment.BaseVoltage.nominalVoltage)

print(results)

{480.0}
